# 03 ML Pipeline: Customer Tribe Discovery

This notebook starts from the prepared parquet outputs created by Notebook 02. It does not repeat raw loading, cleaning, or exploratory data quality work from Notebooks 01 and 02.

The goal is to discover product-first customer tribes from purchase behavior. The client hypothesis is roughly 10-15 tribes, but that range is not used as a modeling constraint. The final recommendation is selected from metrics, stability-ready diagnostics, cluster balance, product/sector lift interpretability, and business usefulness.

## Stage 0: Load Prepared Data

Notebook 03 consumes `df_combined.parquet`, validates the fields required for ML, and merges product metadata only if the prepared file does not already contain it.

In [2]:
from IPython.display import Markdown, display

from pathlib import Path
import sys

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "src").is_dir():
        project_root = candidate
        break
else:
    project_root = Path.cwd().resolve()

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import CONFIG, MODE, DATA_PROCESSED, MODELS, OUTPUTS
from src.data_loader import load_prepared_transactions, peek
from src.utils import set_global_seed

set_global_seed(CONFIG.random_seed)
CONFIG.ensure_directories()

transactions = load_prepared_transactions(cfg=CONFIG)
print(f"Run mode: {MODE}")
print(f"Data path: {DATA_PROCESSED}")
print(f"Model path: {MODELS}")
print(f"Output path: {OUTPUTS}")
peek(transactions, 3)

Run mode: prod
Data path: C:\Users\rothl\Desktop\IE Capstone\carrefour_capstone\data\processed
Model path: C:\Users\rothl\Desktop\IE Capstone\carrefour_capstone\models\prod
Output path: C:\Users\rothl\Desktop\IE Capstone\carrefour_capstone\outputs\prod


idempres,fecha,hora,ticket,cliente,idarticu,unidades,importe,idpromoc,idtiprod,desc_larga_articulo,idsector,desc_sector
i64,date,i64,str,str,i64,i64,f64,str,i64,str,i64,str
7,2022-06-11,1444,"""2022-06-1101780178012008325""","""f9a41c330d2abd575f8990e562feeb…",419547,2,2.8,"""No promo""",1,"""ALBONDIGAS POLLO CARREFOUR 415…",1,"""P.G.C."""
7,2022-04-08,1213,"""2022-04-0800920092027005809""","""4072961ad4f326329c0bbf837905a0…",841133,1,4.69,"""No promo""",2,"""GAMBA PELADA 90/110 ANTONIO Y …",1,"""P.G.C."""
7,2022-02-10,1613,"""2022-02-1000360036007005585""","""dca0afd7eadc32ca02e12e35b55730…",737039,2,1.98,"""No promo""",2,"""ARROZ REDONDO TRADICIONAL BRIL…",1,"""P.G.C."""


## Stage 1: Basket Construction

Each ticket is treated as a basket sentence and each product id is a token. Products are not repeated by quantity unless `baskets.repeat_product_by_quantity` is enabled in the config.

In [ ]:
from src.basket_builder import basket_summary, build_basket_sentences

basket_path = build_basket_sentences(transactions=transactions, cfg=CONFIG)
print(basket_path)
basket_summary(basket_path)

## Stage 2: Item2Vec Product Embeddings

The Word2Vec model learns product proximity from basket co-occurrence. These product embeddings are the core signal used to represent customers.

The training function prints the active hyperparameters and one progress line per epoch. If a cached model already exists, it prints the cache details instead; pass `force=True` to retrain.

In [ ]:
from src.item2vec import save_product_embeddings, train_item2vec

item2vec_model = train_item2vec(basket_path, cfg=CONFIG, verbose=True)
product_embeddings_path = save_product_embeddings(item2vec_model, cfg=CONFIG)
print(product_embeddings_path)

## Stage 3: Product Embedding Validation

Before clustering customers, the nearest-neighbor report checks whether embeddings capture meaningful substitutes, complements, or shared basket missions.

In [ ]:
from src.embedding_validation import validate_product_embeddings

embedding_validation_csv, embedding_validation_md = validate_product_embeddings(
    product_embeddings_path,
    transactions=transactions,
    cfg=CONFIG,
)
print(embedding_validation_csv)
print(embedding_validation_md)

## Stage 4: Customer Embeddings

Customer vectors are weighted means of product embeddings. The default weighting uses spend (`importe`), falling back to quantity and then equal weights when needed. The pipeline aggregates directly to customer level and does not persist row-level embedding joins.

In [ ]:
from src.customer_embeddings import build_customer_embeddings

customer_embeddings_path = build_customer_embeddings(
    product_embeddings_path,
    transactions=transactions,
    cfg=CONFIG,
)
print(customer_embeddings_path)

## Stage 5: Behavioral Features

Behavioral features are stored separately so the analysis can compare embeddings-only segmentation against embeddings plus behavior without allowing KPIs to silently dominate the product signal.

In [ ]:
from src.feature_engineering import build_behavioral_features, build_feature_set

behavior_path = build_behavioral_features(transactions=transactions, cfg=CONFIG)
feature_set_a = build_feature_set(customer_embeddings_path, variant="embeddings_only", cfg=CONFIG)
feature_set_b = build_feature_set(
    customer_embeddings_path,
    behavior_path=behavior_path,
    variant="embeddings_behavior",
    cfg=CONFIG,
)
feature_sets = {
    "embeddings_only": feature_set_a,
    "embeddings_behavior": feature_set_b,
}
print(behavior_path)
print(feature_sets)

## Stage 6: Candidate Model Comparison

The candidate framework compares four approaches without assuming a winner:

- Model A: Raw customer embeddings to Gaussian Mixture Model, evaluated across 6-25 components.
- Model B: Raw customer embeddings to HDBSCAN, allowing the cluster count and noise share to emerge organically.
- Model C: Autoencoder latent representation to Gaussian Mixture Model, evaluated for latent sizes configured in YAML.
- Benchmark: PCA representation to KMeans across 6-25 clusters.

UMAP is reserved for visualization only and is not the primary clustering space.

In [ ]:
from src.model_selection import run_candidate_model_suite

selection_feature_set = CONFIG.get("modeling.feature_set_for_selection", "embeddings_only")
model_suite = run_candidate_model_suite(feature_sets[selection_feature_set], cfg=CONFIG)
model_suite["candidate_results"]

## Stage 7: Tribe Profiling and Interpretability

Every candidate solution is profiled using product lift and sector lift. Raw product popularity is not enough because staples tend to dominate all customers.

In [ ]:
from src.profiling import profile_tribes

profile_paths = {}
for candidate_key, assignment_path in model_suite["assignment_paths"].items():
    profile_paths[candidate_key] = profile_tribes(
        assignment_path,
        transactions=transactions,
        behavior_path=behavior_path,
        cfg=CONFIG,
    )
profile_paths

## Stage 8: Final Model Selection, Exports, and Figures

The comparison table ranks candidates using empirical metrics, cluster balance, assignment confidence where available, HDBSCAN noise share where relevant, and product/sector lift interpretability.

In [ ]:
from src.exports import (
    build_model_comparison,
    export_final_assignments,
    export_final_profiles,
    selected_model,
    write_decision_log,
)
from src.visualization import build_2d_projection_figures, plot_cluster_sizes, plot_model_comparison, plot_top_lifts

comparison_path = build_model_comparison(
    model_suite["candidate_results"],
    profile_paths=profile_paths,
    cfg=CONFIG,
)
selected = selected_model(comparison_path)
selected_key = f"{selected['model_name']}::{selected['model_variant']}"
selected_assignment_path = model_suite["assignment_paths"][selected_key]
selected_profile_path = profile_paths[selected_key]

assignment_export = export_final_assignments(selected_assignment_path, cfg=CONFIG)
profile_export = export_final_profiles(selected_profile_path, cfg=CONFIG)
decision_log_path = write_decision_log(comparison_path, selected_profile_path, cfg=CONFIG)

figures = {
    "model_comparison": plot_model_comparison(comparison_path, cfg=CONFIG),
    "cluster_sizes": plot_cluster_sizes(selected_assignment_path, cfg=CONFIG),
    "projection": build_2d_projection_figures(feature_sets[selection_feature_set], selected_assignment_path, cfg=CONFIG),
    "lift_plots": plot_top_lifts(selected_profile_path, cfg=CONFIG),
}

display(Markdown(f"""
### Selected Solution

- Model: **{selected['model_name']} ({selected['model_variant']})**
- Tribes: **{selected['cluster_count']}**
- Comparison table: `{comparison_path}`
- Assignments: `{assignment_export}`
- Tribe profiles: `{profile_export}`
- Decision log: `{decision_log_path}`
"""))

## Decision Log: Final Tribe Model Selection

This final section is generated after model evaluation and profiling. It records which model was selected, how many tribes were selected, whether that result agrees with the 10-15 tribe client hypothesis, why the selected model won, why alternatives were rejected, what evidence supports the recommendation, remaining limitations, and next production rollout steps.

In [ ]:
from pathlib import Path

display(Markdown(Path(decision_log_path).read_text(encoding="utf-8")))